# Data Prep to create Silver Parquet

## Prerequisites

data_loader -> bronze_chicago_taxi_2024.parquet

In [56]:
import pandas as pd
import requests
from pathlib import Path
import duckdb
import polars as pl
import numpy as np
import geopandas as gpd
from shapely import wkt

TAXI_PATH = "data/bronze_chicago_taxi_2024.parquet"
WEATHER_PATH = "data/bronze_weatherdata.parquet"

SILVER_TAXI_PATH = Path("data/silver_chicago_taxi.parquet")
SILVER_TAXI_PATH.parent.mkdir(parents=True, exist_ok=True)

# Taxi Data

## Plausi Check

column_name                                                        check_name    n_rows     pct severity
                trip_miles                                               <= 0 1338256.0  8.6860     high -> Drop
              trip_seconds                                               <= 0  208815.0  1.3553     high -> Drop
                      fare                                               <= 0   39734.0  0.2579     high -> Drop
                trip_total                                               <= 0   37776.0  0.2452     high -> Drop
                      fare                                            is_null   31816.0  0.2065     high -> See null values
                trip_total                                            is_null   31816.0  0.2065     high -> See null values
              trip_seconds                                               > 6h    8822.0  0.0573     high -> Check via Miles and Fare
              trip_seconds                                            is_null    2936.0  0.0191     high -> See null values
                      fare                                             > 1000    1738.0  0.0113     high -> Check via Seconds and Fare
        trip_end_timestamp                                       parse_failed     345.0  0.0022     high -> Check what happened
                trip_miles                                        > 250 miles     140.0  0.0009     high -> Check via Seconds and Fare
                trip_miles                                            is_null     130.0  0.0008     high -> See null values
      trip_start_timestamp                           after_trip_end_timestamp     100.0  0.0006     high -> Check, but anyways just few
        trip_end_timestamp         not_after_start_when_trip_seconds_gt_15min      32.0  0.0002     high -> Check, but anyways just few
      trip_start_timestamp          not_before_end_when_trip_seconds_gt_15min      32.0  0.0002     high -> Check, but anyways just few
      dropoff_census_tract                                            is_null 8751654.0 56.8032     info
       pickup_census_tract                                            is_null 8547337.0 55.4771     info
              trip_seconds                           < timestamp_diff_seconds 7262520.0 47.1379   medium -> Check
                trip_total       fare + tips + tolls + extras mismatch > 0.01 6948990.0 45.1029   medium -> Check
    dropoff_community_area                                            is_null 1366538.0  8.8696   medium
 dropoff_centroid_latitude                                            is_null 1285413.0  8.3431   medium
 dropoff_centroid_location                                            is_null 1285413.0  8.3431   medium
dropoff_centroid_longitude                                            is_null 1285413.0  8.3431   medium
     pickup_community_area                                            is_null  429216.0  2.7859   medium
  pickup_centroid_latitude                                            is_null  421354.0  2.7348   medium
  pickup_centroid_location                                            is_null  421354.0  2.7348   medium
 pickup_centroid_longitude                                            is_null  421354.0  2.7348   medium
             fare_per_mile                                               > 50  200919.0  1.3041   medium
            total_per_mile                                              > 100  146833.0  0.9530   medium
                      tips                                    tip_rate > 100%   80542.0  0.5228   medium
                    extras                                            is_null   31816.0  0.2065   medium
                      tips                                            is_null   31816.0  0.2065   medium
                     tolls                                            is_null   31816.0  0.2065   medium
              trip_seconds                                               > 3h   14920.0  0.0968   medium
                       mph                                          > 100 mph    4607.0  0.0299   medium
                      fare                                              > 500    3304.0  0.0214   medium
                    extras                                              > 100    2078.0  0.0135   medium
                trip_total                                             > 1000    1791.0  0.0116   medium
                trip_miles                                        > 100 miles    1243.0  0.0081   medium
              trip_seconds abs(trip_seconds - timestamp_diff_seconds) > 15min     138.0  0.0009   medium
                     tolls                                              > 100      30.0  0.0002   medium
                      tips                                              > 200      11.0  0.0001   medium

In [57]:
# drop / filter conditions from plausi-check
# Assumptions what defines a valid trip in our understanding:
valid_trip_miles = (pl.col("trip_miles") > 0) & (pl.col("trip_miles") <= 250) # valid trip: 0 < trip_miles <= 250
valid_trip_seconds = (pl.col("trip_seconds") > 59) & (pl.col("trip_seconds") <= 10_800) # valid trip: 59 < trip_seconds < 10_800 (3 hours)
valid_trip_fare = (pl.col("fare") > 0) # valid trip: fare > 0
valid_trip_tips = pl.col("tips") >= 0 # valid trip: tips >= 0
valid_trip_tolls = pl.col("tolls") >= 0 # valid trip: tolls >= 0
valid_trip_extras = pl.col("extras") >= 0 # valid trip: extra >= 0
valid_trip_total = (pl.col("trip_total") > 0) # valid trip : total > 0

Drop columns and rows with null values

In [58]:

DROP_COLUMNS = [
    ":created_at", # meta data from API
    ":updated_at", # meta data from API
    ":version", # meta data from API
    ":id", # meta data from API
    "dropoff_community_area", # census tract is more granular
    "pickup_community_area" # census tract is more granular
]

REQUIRED_NOT_NULL_COLUMNS = [
    "pickup_centroid_latitude",
    "pickup_centroid_longitude",
    "pickup_centroid_location",
    "dropoff_centroid_latitude",
    "dropoff_centroid_longitude",
    "dropoff_centroid_location"
]

taxi_silver = (
    pl.scan_parquet(TAXI_PATH)
    .drop(DROP_COLUMNS)
    .filter(
        pl.all_horizontal([
            pl.col(col).is_not_null()
            for col in REQUIRED_NOT_NULL_COLUMNS
        ])
    )
)

In [59]:
# Drop rows by conditions
taxi_silver = taxi_silver.filter(
    valid_trip_miles
    & valid_trip_seconds
    & valid_trip_fare
    & valid_trip_tips
    & valid_trip_tolls
    & valid_trip_extras
    & valid_trip_total 
)

Get census_tract from coordinates

In [60]:
# Step 1: get coords of missing census tracts
unique_missing_coords_df = duckdb.sql(f"""
    WITH pickup AS (
        SELECT DISTINCT
            pickup_centroid_latitude AS lat,
            pickup_centroid_longitude AS lon
        FROM read_parquet('{SILVER_TAXI_PATH}')
        WHERE pickup_census_tract IS NULL
          AND pickup_centroid_latitude IS NOT NULL
          AND pickup_centroid_longitude IS NOT NULL
    ),

    dropoff AS (
        SELECT DISTINCT
            dropoff_centroid_latitude AS lat,
            dropoff_centroid_longitude AS lon
        FROM read_parquet('{SILVER_TAXI_PATH}')
        WHERE dropoff_census_tract IS NULL
          AND dropoff_centroid_latitude IS NOT NULL
          AND dropoff_centroid_longitude IS NOT NULL
    )

    SELECT DISTINCT
        lat,
        lon
    FROM (
        SELECT * FROM pickup
        UNION ALL
        SELECT * FROM dropoff
    )
    ORDER BY lat, lon
""").df()

print(unique_missing_coords_df)
print(f"Unique coordinates: {len(unique_missing_coords_df)}")

# Step 2: get polynom of census tracts
CENSUS_TRACTS_CSV_PATH = "raw_data/Census_Tracts_20260526.csv"
GEOM_COL = "the_geom"
TRACT_ID_COL = "CENSUS_T_1" 
tracts_df = pd.read_csv(CENSUS_TRACTS_CSV_PATH)

tracts_df[GEOM_COL] = tracts_df[GEOM_COL].apply(wkt.loads)

tracts_gdf = gpd.GeoDataFrame(
    tracts_df,
    geometry=GEOM_COL,
    crs="EPSG:4326",
)

points_gdf = gpd.GeoDataFrame(
    unique_missing_coords_df,
    geometry=gpd.points_from_xy(
        unique_missing_coords_df["lon"],
        unique_missing_coords_df["lat"],
    ),
    crs="EPSG:4326",
)

tracts_gdf = tracts_gdf[[TRACT_ID_COL, GEOM_COL]].rename(
    columns={
        TRACT_ID_COL: "census_tract_filled",
        GEOM_COL: "geometry",
    }
)

tracts_gdf = gpd.GeoDataFrame(
    tracts_gdf,
    geometry="geometry",
    crs="EPSG:4326",
)

point_to_tract_df = gpd.sjoin(
    points_gdf,
    tracts_gdf,
    how="left",
    predicate="within",
)

point_to_tract_df = (
    point_to_tract_df
    .drop(columns=["geometry", "index_right"])
    .reset_index(drop=True)
)

print(point_to_tract_df)
print("Unmatched points:", point_to_tract_df["census_tract_filled"].isna().sum())

# Use mapping df to fill values in taxi data


taxi = taxi_silver
mapping = pl.from_pandas(point_to_tract_df).lazy()

pickup_mapping = (
    mapping
    .select([
        pl.col("lat").alias("pickup_centroid_latitude"),
        pl.col("lon").alias("pickup_centroid_longitude"),
        pl.col("census_tract_filled").alias("pickup_census_tract_from_geo"),
    ])
)

dropoff_mapping = (
    mapping
    .select([
        pl.col("lat").alias("dropoff_centroid_latitude"),
        pl.col("lon").alias("dropoff_centroid_longitude"),
        pl.col("census_tract_filled").alias("dropoff_census_tract_from_geo"),
    ])
)

taxi_filled = (
    taxi
    .join(
        pickup_mapping,
        on=["pickup_centroid_latitude", "pickup_centroid_longitude"],
        how="left",
    )
    .join(
        dropoff_mapping,
        on=["dropoff_centroid_latitude", "dropoff_centroid_longitude"],
        how="left",
    )
    .with_columns([
        pl.coalesce([
            pl.col("pickup_census_tract"),
            pl.col("pickup_census_tract_from_geo"),
        ]).alias("pickup_census_tract"),

        pl.coalesce([
            pl.col("dropoff_census_tract"),
            pl.col("dropoff_census_tract_from_geo"),
        ]).alias("dropoff_census_tract"),
    ])
    .drop([
        "pickup_census_tract_from_geo",
        "dropoff_census_tract_from_geo",
    ])
)

          lat        lon
0   41.660136 -87.602848
1   41.663671 -87.540936
2   41.673820 -87.635740
3   41.689730 -87.669054
4   41.690633 -87.570058
..        ...        ...
72  41.986712 -87.663416
73  41.993930 -87.758354
74  42.001571 -87.695013
75  42.007613 -87.813781
76  42.009623 -87.670167

[77 rows x 2 columns]
Unique coordinates: 77
          lat        lon  census_tract_filled
0   41.660136 -87.602848          17031540100
1   41.663671 -87.540936          17031550100
2   41.673820 -87.635740          17031530500
3   41.689730 -87.669054          17031750500
4   41.690633 -87.570058          17031510400
..        ...        ...                  ...
72  41.986712 -87.663416          17031030500
73  41.993930 -87.758354          17031120200
74  42.001571 -87.695013          17031020600
75  42.007613 -87.813781          17031090200
76  42.009623 -87.670167          17031010700

[77 rows x 3 columns]
Unmatched points: 0


In [61]:
taxi_filled.sink_parquet(SILVER_TAXI_PATH)

print(f"Silver taxi parquet written to: {SILVER_TAXI_PATH}")

Silver taxi parquet written to: data/silver_chicago_taxi.parquet


## Null Values

column_name                 n_rows   n_nulls  null_pct
      dropoff_census_tract 15406960 8751654.0     56.80 -> Ok, might be estimated via drop off location or anyways not needed because of hexagon structure
       pickup_census_tract 15406960 8547337.0     55.48 -> Ok, might be estimated via drop off location or anyways not needed because of hexagon structure
    dropoff_community_area 15406960 1366538.0      8.87 -> Ok, might be estimated via drop off location or anyways not needed because of hexagon structure
 dropoff_centroid_location 15406960 1285413.0      8.34 -> drop, only very few might be estimated based on census_tract, Note: one could consider leaving datasets with either dropoff or pickup as it could be used as info for hexagon ("how many pick ups in hour X in hexagon Y?")
 dropoff_centroid_latitude 15406960 1285413.0      8.34 -> drop, only very few might be estimated based on census_tract 
dropoff_centroid_longitude 15406960 1285413.0      8.34 -> drop, only very few might be estimated based on census_tract 
     pickup_community_area 15406960  429216.0      2.79 -> Ok, might be estimated via drop off location or anyways not needed because of hexagon structure
  pickup_centroid_latitude 15406960  421354.0      2.73 -> drop, only very few might be estimated based on census_tract 
 pickup_centroid_longitude 15406960  421354.0      2.73 -> drop, only very few might be estimated based on census_tract 
  pickup_centroid_location 15406960  421354.0      2.73 -> -> drop, only very few might be estimated based on census_tract 
                      fare 15406960   31816.0      0.21 -> Estimate, average fare for "trip_seconds" or more precise some range of it
                      tips 15406960   31816.0      0.21 -> Estimate, average tips for "trip_seconds" or more precise some range of it
                     tolls 15406960   31816.0      0.21 -> Estimate, average tolls for "trip_seconds" or more precise some range of it
                    extras 15406960   31816.0      0.21 -> Estimate, average extras for "trip_seconds" or more precise some range of it
                trip_total 15406960   31816.0      0.21 -> Estimate, sum of fare, tips, tolls, extras
              trip_seconds 15406960    2936.0      0.02 -> Estimate, "trip_end_timestamp" - "trip_start_timestamp" (if both are not null)
                   trip_id 15406960       0.0      0.00 -> Ok
                   taxi_id 15406960       0.0      0.00 -> Ok
      trip_start_timestamp 15406960       0.0      0.00 -> Ok
        trip_end_timestamp 15406960     345.0      0.00 -> Estimate, "trip_start_timestamp" + "trip_seconds" (if "trip_seconds" not null)
                trip_miles 15406960     130.0      0.00 -> Estimate, average trip_miles for "trip_seconds" or more precise some range of it
               :created_at 15406960       0.0      0.00 -> Ok, drop anyways
               :updated_at 15406960       0.0      0.00 -> Ok, drop anyways
                  :version 15406960       0.0      0.00 -> Ok, drop anyways
              payment_type 15406960       0.0      0.00 -> Ok
                   company 15406960       0.0      0.00 -> Ok
                       :id 15406960       0.0      0.00 -> Ok, drop anyways

### Quality Check

In [62]:
duckdb.sql(f"""
    SELECT COUNT(*)
    FROM read_parquet('{SILVER_TAXI_PATH}')  
""").show()

con = duckdb.connect()

columns = con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{SILVER_TAXI_PATH}')
""").df()["column_name"].tolist()

parts = []

for col in columns:
    parts.append(f"""
    SELECT
        '{col}' AS column_name,
        COUNT(*) AS n_rows,
        SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) AS n_nulls,
        ROUND(
            100.0 * SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS null_pct
    FROM read_parquet('{SILVER_TAXI_PATH}')
    """)

query = "\nUNION ALL\n".join(parts) + "\nORDER BY null_pct DESC"

null_report = con.sql(query).df()
print(null_report.to_string(index=False))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     12662309 │
└──────────────┘

               column_name   n_rows  n_nulls  null_pct
                   taxi_id 12662309      0.0       0.0
                   trip_id 12662309      0.0       0.0
      trip_start_timestamp 12662309      0.0       0.0
        trip_end_timestamp 12662309      0.0       0.0
              trip_seconds 12662309      0.0       0.0
                trip_miles 12662309      0.0       0.0
       pickup_census_tract 12662309      0.0       0.0
      dropoff_census_tract 12662309      0.0       0.0
                      fare 12662309      0.0       0.0
                      tips 12662309      0.0       0.0
                     tolls 12662309      0.0       0.0
                    extras 12662309      0.0       0.0
                trip_total 12662309      0.0       0.0
              payment_type 12662309      0.0       0.0
                   company 12662309      0.0       0.0
  pickup_centroid

In [41]:
duckdb.sql(f"""
    SELECT distinct pickup_centroid_latitude, pickup_centroid_longitude
    FROM read_parquet('{SILVER_TAXI_PATH}')  
    WHERE pickup_census_tract is null      
""").show()

duckdb.sql(f"""
    SELECT distinct dropoff_centroid_latitude, dropoff_centroid_longitude
    FROM read_parquet('{SILVER_TAXI_PATH}')  
    WHERE dropoff_census_tract is null      
""").show()

┌──────────────────────────┬───────────────────────────┐
│ pickup_centroid_latitude │ pickup_centroid_longitude │
│          double          │          double           │
├──────────────────────────┼───────────────────────────┤
│             41.927260956 │             -87.765501609 │
│             41.947791586 │             -87.683834942 │
│             41.953582125 │              -87.72345239 │
│             41.944226601 │             -87.655998182 │
│              41.82371281 │             -87.602350437 │
│             41.763246799 │             -87.616134111 │
│             41.794090253 │             -87.592310855 │
│              41.77887686 │             -87.594925439 │
│             41.706125752 │             -87.598255838 │
│             41.954027649 │             -87.763399032 │
│                   ·      │                   ·       │
│                   ·      │                   ·       │
│                   ·      │                   ·       │
│             41.922686284 │   

# Weather Data

## Null Values
column_name  n_rows  n_nulls  null_pct
       gust   24360  18401.0     75.54 -> drop, too many missing values and likely no value add as wind speed is available
      skyc3   24360  18227.0     74.82 -> drop, too many missing values and likely no value add as cloud cover level 1 is available
      skyc2   24360  11411.0     46.84 -> drop, too many missing values and likely no value add as cloud cover level 1 is available
       p01m   24360      3.0      0.01 -> interpolate
       relh   24360      2.0      0.01 -> interpolate
       tmpc   24360      2.0      0.01 -> interpolate
    station   24360      0.0      0.00 -> Ok
      skyc1   24360      0.0      0.00 -> Ok
        lon   24360      0.0      0.00 -> Ok
      valid   24360      0.0      0.00 -> Ok
       vsby   24360      0.0      0.00 -> Ok
       sknt   24360      1.0      0.00 -> interpolate
        lat   24360      0.0      0.00 -> Ok

# Data Prep to create Gold Parquet

## Taxi Data
Generate two datasets -> 1. Gold with granular data & 2. Gold with hourly and hexagon data
Add features

## Weather Data

Create hourly indexed dataset for weather data from 01.01.2024 to 24.05.2026

In [27]:
SILVER_WEATHER_PATH = Path(WEATHER_PATH)
GOLD_WEATHER_PATH = Path("data/gold_weather_hourly.parquet")

START_TS = "2024-01-01 00:00:00"
END_TS = "2026-05-01 23:00:00" # also max date in taxi data


def mode_or_na(series: pd.Series):
    """Return most frequent non-null value, otherwise NA."""
    mode_values = series.dropna().mode()
    if len(mode_values) == 0:
        return pd.NA
    return mode_values.iloc[0]


def create_hourly_weather_gold(
    df: pd.DataFrame,
    start_ts: str = START_TS,
    end_ts: str = END_TS,
) -> pd.DataFrame:
    """
    Create hourly gold weather dataset.

    Steps:
    - Parse valid timestamp
    - Restrict to requested date range
    - Aggregate observations to hourly level
    - Create complete hourly timestamp spine
    - Reindex to all hours
    - Linearly interpolate numeric weather columns
    - Fill categorical/context columns
    - Add date/hour helper columns
    """

    df = df.copy()

    # 1. Basic cleanup
    df["valid"] = pd.to_datetime(df["valid"], utc=True)
    df = df.dropna(subset=["valid"])
    
    # Convert UTC to chicago timezone
    df["valid"] = df["valid"].dt.tz_convert("America/Chicago")

    start_ts = (
        pd.Timestamp(start_ts)
        .tz_localize(
            "America/Chicago",
            ambiguous=False, # takes winter time hour
            nonexistent="shift_forward"
        )
    )

    end_ts = (
        pd.Timestamp(end_ts)
        .tz_localize(
            "America/Chicago",
            ambiguous=False, # takes winter time hour
            nonexistent="shift_forward"
        )
    )

    # Optional: keep only relevant date range
    df = df[(df["valid"] >= start_ts) & (df["valid"] <= end_ts)]

    # 2. Create hourly timestamp
    df["valid_hour"] = df["valid"].dt.floor("h", ambiguous=False, nonexistent="shift_forward")

    # 3. Cast numeric columns
    numeric_cols = [
        "tmpc",   # temperature Celsius
        "relh",   # relative humidity
        "sknt",   # wind speed in knots
        "p01m",   # precipitation
        "vsby",   # visibility
        "lat",
        "lon",
    ]

    existing_numeric_cols = [col for col in numeric_cols if col in df.columns]

    for col in existing_numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 4. Define aggregation rules
    agg_dict = {}

    # Numeric weather values: hourly mean
    for col in ["tmpc", "relh", "sknt", "vsby"]:
        if col in df.columns:
            agg_dict[col] = "mean"

    # Precipitation: hourly sum is usually more sensible than mean
    if "p01m" in df.columns:
        agg_dict["p01m"] = "sum"

    # Station/location fields
    if "station" in df.columns:
        agg_dict["station"] = mode_or_na

    if "lat" in df.columns:
        agg_dict["lat"] = "mean"

    if "lon" in df.columns:
        agg_dict["lon"] = "mean"

    # Categorical weather condition
    if "skyc1" in df.columns:
        agg_dict["skyc1"] = mode_or_na

    # 5. Aggregate to hourly level
    hourly = (
        df
        .groupby("valid_hour", as_index=True)
        .agg(agg_dict)
        .sort_index()
    )

    # 6. Complete hourly index from start to end
    full_hourly_index = pd.date_range(
        start=start_ts,
        end=end_ts,
        freq="h",
        name="valid_hour",
    )

    hourly = hourly.reindex(full_hourly_index)

    # 7. Interpolate numeric columns over missing hours
    numeric_interpolate_cols = [
        col for col in ["tmpc", "relh", "sknt", "p01m", "vsby", "lat", "lon"]
        if col in hourly.columns
    ]

    hourly[numeric_interpolate_cols] = (
        hourly[numeric_interpolate_cols]
        .interpolate(method="time", limit_direction="both")
    )

    # 8. Fill categorical/context columns
    categorical_fill_cols = [
        col for col in ["station", "skyc1"]
        if col in hourly.columns
    ]

    for col in categorical_fill_cols:
        hourly[col] = hourly[col].ffill().bfill()

    # 9. Back to normal dataframe
    gold = hourly.reset_index()

    # 10. Add helper columns for joins / ML
    # TODO might drop some later if not used for joining
    gold["date"] = gold["valid_hour"].dt.date
    gold["year"] = gold["valid_hour"].dt.year
    gold["month"] = gold["valid_hour"].dt.month
    gold["day"] = gold["valid_hour"].dt.day
    gold["hour"] = gold["valid_hour"].dt.hour
    gold["weekday"] = gold["valid_hour"].dt.weekday

    # 11. Optional quality flags
    original_hours = set(df["valid_hour"].dropna().unique())

    gold["was_observed_hour"] = gold["valid_hour"].isin(original_hours)
    gold["was_interpolated_hour"] = ~gold["was_observed_hour"]
    
    # Onehot encoding
    skyc1_dummies = pd.get_dummies(
        gold["skyc1"],
        prefix="skyc1",
        dummy_na=False,
        dtype="int8",
    )

    gold = pd.concat([gold, skyc1_dummies], axis=1)
    
    # Drop unused columns
    gold = gold.drop(columns=["skyc1", "station", "lat", "lon"])

    return gold


GOLD_WEATHER_PATH.parent.mkdir(parents=True, exist_ok=True)

weather_silver = pd.read_parquet(SILVER_WEATHER_PATH)

weather_gold = create_hourly_weather_gold(
    weather_silver,
    start_ts=START_TS,
    end_ts=END_TS,
)

weather_gold.to_parquet(
    GOLD_WEATHER_PATH,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

print(f"Gold weather parquet written to: {GOLD_WEATHER_PATH}")
print(f"Rows: {len(weather_gold):,}")
print(f"Start: {weather_gold['valid_hour'].min()}")
print(f"End: {weather_gold['valid_hour'].max()}")
print()
print(weather_gold.head().to_string(index=False))
print()
print(weather_gold.tail().to_string(index=False))

Gold weather parquet written to: data/gold_weather_hourly.parquet
Rows: 20,447
Start: 2024-01-01 00:00:00-06:00
End: 2026-05-01 23:00:00-05:00

               valid_hour  tmpc  relh  sknt  vsby   p01m       date  year  month  day  hour  weekday  was_observed_hour  was_interpolated_hour  skyc1_BKN  skyc1_CLR  skyc1_FEW  skyc1_OVC  skyc1_SCT  skyc1_VV 
2024-01-01 00:00:00-06:00  1.11 75.26  12.0  10.0 0.0001 2024-01-01  2024      1    1     0        0               True                  False          0          0          0          1          0          0
2024-01-01 01:00:00-06:00  1.11 75.26  12.0   8.0 0.0001 2024-01-01  2024      1    1     1        0               True                  False          0          0          0          1          0          0
2024-01-01 02:00:00-06:00  0.56 81.63  10.0   7.0 0.0001 2024-01-01  2024      1    1     2        0               True                  False          0          0          0          1          0          0
2024-01-01 03:00:00-

In [26]:
duckdb.sql(f"""
    SELECT hour, count(*)
    FROM  read_parquet('{GOLD_WEATHER_PATH}')
    GROUP BY hour
    ORDER BY hour asc       
           """).show(max_rows=100)

duckdb.sql(f"""
    SELECT date, hour, was_interpolated_hour
    FROM  read_parquet('{GOLD_WEATHER_PATH}')
    WHERE was_interpolated_hour = True    
           """).show(max_rows=100)

┌───────┬──────────────┐
│ hour  │ count_star() │
│ int32 │    int64     │
├───────┼──────────────┤
│     0 │          875 │
│     1 │          877 │
│     2 │          872 │
│     3 │          875 │
│     4 │          875 │
│     5 │          875 │
│     6 │          875 │
│     7 │          875 │
│     8 │          875 │
│     9 │          875 │
│    10 │          875 │
│    11 │          875 │
│    12 │          875 │
│    13 │          875 │
│    14 │          875 │
│    15 │          875 │
│    16 │          875 │
│    17 │          875 │
│    18 │          875 │
│    19 │          875 │
│    20 │          875 │
│    21 │          875 │
│    22 │          875 │
│    23 │          875 │
└───────┴──────────────┘
  24 rows    2 columns

┌────────────┬───────┬───────────────────────┐
│    date    │ hour  │ was_interpolated_hour │
│    date    │ int32 │        boolean        │
├────────────┼───────┼───────────────────────┤
│ 2024-11-03 │     1 │ true                  │
│ 2025-03-09 │  